# P66 — Una lógica orientada a máquina basada en el principio de resolución

## 1. Título y paper

**Paper:** *A Machine-Oriented Logic Based on the Resolution Principle*  
**Autoría:** J. A. Robinson  
**Año y venue:** 1965 · Journal of the ACM, 12(1), 23–41  
**Nivel:** L3 · **Motor:** `resolucion`  
**Ficha completa:** [`P66_resolucion`](../../papers/foundational/P66_resolucion/README.md)

**Hito:** Reduce toda la inferencia de primer orden a una sola regla, y hace la unificación computable con el unificador más general.

- [doi:10.1145/321250.321253](https://doi.org/10.1145/321250.321253)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los cálculos lógicos existentes tenían muchas reglas pensadas para el razonamiento humano. Aplicarlas a máquina generaba una explosión de caminos sin criterio.
2. Ejecutar una implementación mínima de la propuesta: Una única regla —la resolución— sobre cláusulas, junto con el algoritmo de unificación que calcula el unificador más general: la sustitución mínima que iguala dos términos sin comprometer nada de más.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P65
- Herbrand (1930), teorema de Herbrand
- Prawitz (1960), unificación implícita


## 4. Intuición

Dos frases que se contradicen en un punto se pueden fundir en una tercera que ya no menciona ese punto. Repite hasta llegar a la nada —la cláusula vacía— y habrás demostrado que las premisas eran incompatibles. Para que funcione con variables hace falta saber igualarlas: eso es la unificación.


## 5. Concepto mínimo

```text
Resolución:  de (A ∨ L)  y  (B ∨ ¬L')  con σ = mgu(L, L')
             se sigue  (A ∨ B)σ

Unificador más general (mgu): la sustitución MÍNIMA que iguala dos términos
    Humano(x)      y  Humano(Sócrates)   →  {x = Sócrates}
    Padre(x, y)    y  Padre(Juan, z)     →  {x = Juan, y = z}
    Humano(Sócrates) y Humano(Platón)    →  no unifican
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('resolucion', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuáles de los cuatro pares de términos unifican?
2. ¿Qué liga el unificador en `Padre(x,y)` con `Padre(Juan,z)`?
3. ¿Con qué se cierra una demostración por refutación?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('resolucion', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('resolucion', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Tres de los cuatro pares unifican; «Sócrates» y «Platón» son constantes distintas y ninguna sustitución las iguala. El unificador de `Padre` liga `x = Juan` y deja `y = z` sin resolver: no compromete nada de más. Y la refutación cierra con **la cláusula vacía**, que es la contradicción.


## 10. Comentario pedagógico

Este resultado es la base de Prolog y de todos los demostradores automáticos. La idea de negar la conclusión y buscar contradicción es además el patrón que reaparece en la verificación formal moderna: no se demuestra que algo es cierto, se demuestra que su negación es imposible.


## 11. Error o anti-patrón deliberado

Anti-patrón: olvidar la comprobación de ocurrencia al unificar.


In [ ]:
print('Unificar x con f(x) daria x = f(f(f(...))), un termino infinito.')
print('La comprobacion de ocurrencia lo impide, y cuesta tiempo.')
print('Muchos Prolog la omiten por velocidad: es correcto y es un riesgo declarado.')

## 12. Corrección

Qué demuestra la refutación y qué no:


In [ ]:
r = run_paper_lab('resolucion', seed=7)['result']
print('cierra con clausula vacia :', r['cierra_por_refutacion'])
print('regla unica               :', r['regla_unica'])
print('Demostrar que Socrates es mortal = negarlo y llegar a contradiccion.')

## 13. Desafío guiado

Revisa los cuatro casos de unificación y explica, para el que falla, por qué ninguna sustitución puede arreglarlo.


In [ ]:
r = run_paper_lab('resolucion', seed=3)['result']
show(r)

## 14. Desafío autónomo

Escribe en cláusulas un dominio pequeño de tu elección —tres o cuatro hechos y dos reglas— y demuestra una conclusión por refutación a mano. Después comprueba qué pasa si la conclusión NO se sigue: ¿termina el procedimiento?


## 15. Evidencia de aprendizaje

Guarda la tabla de unificación con sus unificadores y tu explicación de por qué la refutación cierra con el vacío.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P66_resolucion/README.md) · evaluación formal: [`assessments/papers/P66_resolucion.md`](../../assessments/papers/P66_resolucion.md)


## 16. Cierre

Ya se puede deducir. Falta decidir por dónde buscar cuando hay muchos caminos posibles y todos son válidos: eso lo resuelve una heurística con garantía.


## 17. Conexión con el siguiente hito

- P69
- P71

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
